# LeetCode 1894 — Find the Student that Will Replace the Chalk bằng Python

Link đề: https://leetcode.com/problems/find-the-student-that-will-replace-the-chalk/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search, Simulation, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def chalkReplacer(self, chalk: List[int], k: int) -> int:
        ...
```

Lộ trình:
1. **Mô phỏng từng học sinh** cho tới khi hết phấn (`O(k / min(chalk))` bước — có thể tới `10⁹`).
2. **Bỏ qua các vòng đầy đủ** bằng `k mod Σ chalk`, rồi mô phỏng **một** vòng (`O(n)`).
3. **Tổng tiền tố + tìm kiếm nhị phân** (`bisect_right`) trên vòng cuối (`O(n)` tiền xử lý, `O(log n)` mỗi truy vấn).
4. **Một lượt "trừ dần" có dừng sớm** — không cần tính tổng nếu phấn hết ngay trong vòng đầu.
5. Mở rộng: trả lời **nhiều truy vấn `k`** trên cùng mảng `chalk`.

## 1. Mục tiêu học

1. Nhận ra **tính tuần hoàn** của quá trình: sau mỗi vòng đầy đủ, trạng thái lặp lại với `k` giảm đúng `Σ chalk` ⇒ dùng **phép chia lấy dư** để nhảy qua mọi vòng.
2. Viết mô phỏng một vòng chính xác với điều kiện "**nhỏ hơn nghiêm ngặt**" (`k < chalk[i]` thì học sinh `i` phải thay phấn).
3. Dùng **tổng tiền tố** và `bisect_right` để tìm học sinh đầu tiên có tổng tiền tố **vượt** `k`.
4. Hiểu khi nào cần chú ý **tràn số** (tổng tới `10¹⁰` trong C++/Java) — Python không lo.
5. Kiểm chứng bằng mô phỏng trực tiếp với `k` nhỏ.

## 2. Đề bài & ví dụ

Lớp có `n` học sinh đánh số `0..n−1`. Giáo viên lần lượt giao bài cho học sinh `0, 1, …, n−1`, rồi **quay lại** học sinh `0` và tiếp tục.

Ban đầu có `k` viên phấn. Học sinh `i` khi được giao bài dùng `chalk[i]` viên. Nếu số phấn hiện tại **nhỏ hơn nghiêm ngặt** `chalk[i]`, học sinh `i` phải đi **thay phấn**.

Trả về chỉ số học sinh sẽ đi thay phấn.

**Ví dụ 1:** `chalk = [5,1,5], k = 22` → `0`. Phấn còn: `17, 16, 11, 6, 5, 0`; đến học sinh 0 cần 5 > 0 ⇒ học sinh 0.

**Ví dụ 2:** `chalk = [3,4,1,2], k = 25` → `1`. Phấn còn: `22, 18, 17, 15, 12, 8, 7, 5, 2`; học sinh 1 cần 4 > 2 ⇒ học sinh 1.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `1 ≤ chalk[i] ≤ 10⁵`, `1 ≤ k ≤ 10⁹`.

In [1]:
import random
import time
from bisect import bisect_right
from itertools import accumulate
from typing import List

examples = [
    (([5, 1, 5], 22), 0),
    (([3, 4, 1, 2], 25), 1),
]
for (chalk, k), want in examples:
    print(f"chalk={chalk}, k={k} -> kỳ vọng {want}")

chalk=[5, 1, 5], k=22 -> kỳ vọng 0
chalk=[3, 4, 1, 2], k=25 -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1 Phép chia lấy dư và tính tuần hoàn

Sau một vòng đầy đủ, `k` giảm `S = Σ chalk` và thứ tự học sinh lặp lại. Vậy số phấn khi bắt đầu vòng "cuối" là `k mod S` (nếu `k mod S` vẫn đủ cho cả vòng thì đã có thêm một vòng đầy đủ — mâu thuẫn).

In [2]:
chalk, k = [5, 1, 5], 22
S = sum(chalk)
print(f"S = {S}, số vòng đầy đủ = {k // S}, phấn khi vào vòng cuối = {k % S}")

S = 11, số vòng đầy đủ = 2, phấn khi vào vòng cuối = 0


### 3.2 Tổng tiền tố

`pre[i] = chalk[0] + … + chalk[i]`. Học sinh `i` (trong vòng cuối, phấn ban đầu `r`) phải thay phấn ⇔ `pre[i−1] ≤ r < pre[i]` ⇔ `i` là chỉ số **đầu tiên** có `pre[i] > r`.

In [3]:
pre = list(accumulate([3, 4, 1, 2]))
print("pre =", pre)

pre = [3, 7, 8, 10]


### 3.3 `bisect_right`

`bisect_right(pre, r)` = số phần tử `≤ r` = chỉ số đầu tiên có `pre[i] > r` (vì `pre` tăng nghiêm ngặt khi `chalk[i] ≥ 1`).

In [4]:
pre = [3, 7, 8, 10]
print([(r, bisect_right(pre, r)) for r in [0, 2, 3, 6, 7, 9]])

[(0, 0), (2, 0), (3, 1), (6, 1), (7, 2), (9, 3)]


### 3.4 Tràn số trong ngôn ngữ khác

`Σ chalk ≤ 10⁵ · 10⁵ = 10¹⁰ > 2³¹ − 1`. Trong C++/Java dùng `long`. Python `int` tùy ý.

In [5]:
print(10**5 * 10**5, ">", 2**31 - 1)

10000000000 > 2147483647


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2 `chalk = [3,4,1,2]`, `k = 25`, `S = 10`:
- Hai vòng đầy đủ tiêu `20` viên ⇒ còn `25 mod 10 = 5`.
- Vòng cuối: học sinh 0 dùng 3 (còn 2), học sinh 1 cần 4 > 2 ⇒ **học sinh 1**.

Bằng tiền tố: `pre = [3, 7, 8, 10]`, chỉ số đầu tiên có `pre[i] > 5` là `1`. ✓

Ví dụ 1: `S = 11`, `22 mod 11 = 0` ⇒ học sinh đầu tiên có `pre[i] > 0` là `0`. ✓ (Lưu ý: còn đúng 0 viên, học sinh 0 cần 5 > 0.)

## 5. Cách 1 — Brute Force / baseline: mô phỏng từng học sinh

In [6]:
def chalk_simulate(chalk, k):
    n = len(chalk)
    i = 0
    while k >= chalk[i]:
        k -= chalk[i]
        i = (i + 1) % n
    return i

for (chalk, k), want in examples:
    print(chalk_simulate(chalk, k), "kỳ vọng", want)

0 kỳ vọng 0
1 kỳ vọng 1


**Độ phức tạp:** số bước `≈ k / trung bình(chalk)`; worst `chalk = [1]*n`, `k = 10⁹` ⇒ `10⁹` bước — quá chậm. Space `O(1)`.

**Nút thắt:** lặp lại cùng một vòng hàng triệu lần.

In [7]:
for k in [10**5, 10**6]:
    t0 = time.perf_counter(); chalk_simulate([1, 1, 1, 1], k)
    print(f"k={k}: {time.perf_counter() - t0:.3f}s (k = 1e9 sẽ chậm hơn ~1000 lần)")

k=100000: 0.004s (k = 1e9 sẽ chậm hơn ~1000 lần)
k=1000000: 0.037s (k = 1e9 sẽ chậm hơn ~1000 lần)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight (tuần hoàn):** gọi `S = Σ chalk`. Nếu `k ≥ S` thì cả lớp đều đủ phấn trong vòng này và vòng sau bắt đầu với `k − S`. Lặp lại ⇒ vòng cuối bắt đầu với `r = k mod S` (vì `r < S`, sẽ có người thiếu phấn trong vòng này).

**Insight 2 (tiền tố):** trong vòng cuối, học sinh `i` thiếu ⇔ `pre[i] > r` lần đầu.

**Quy tắc nếu… thì…:**
- `r = k % S`.
- Duyệt `i`: nếu `r < chalk[i]` thì trả `i`; ngược lại `r −= chalk[i]`.
- Vòng lặp chắc chắn dừng trong vòng cuối vì `r < S`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `k mod S` + mô phỏng một vòng

In [8]:
def chalk_mod(chalk, k):
    r = k % sum(chalk)
    for i, c in enumerate(chalk):
        if r < c:
            return i
        r -= c
    return -1          # không xảy ra vì r < sum(chalk)

for (chalk, k), want in examples:
    print(chalk_mod(chalk, k), "kỳ vọng", want)

0 kỳ vọng 0
1 kỳ vọng 1


Code nộp LeetCode của Cách 2: xem mục 9.

**Độ phức tạp:** Time `O(n)` (một lượt tính tổng + tối đa một vòng), Space `O(1)`.

### 7.2 Cách 3 — Tổng tiền tố + `bisect_right`

In [9]:
def chalk_bisect(chalk, k):
    pre = list(accumulate(chalk))
    r = k % pre[-1]
    return bisect_right(pre, r)

for (chalk, k), want in examples:
    print(chalk_bisect(chalk, k), "kỳ vọng", want)

0 kỳ vọng 0
1 kỳ vọng 1


**Độ phức tạp:** tiền xử lý `O(n)`, truy vấn `O(log n)`, Space `O(n)`. Có lợi khi **nhiều** truy vấn `k`.

### 7.3 Cách 4 — Một lượt có dừng sớm (không cần `sum` khi hết phấn ngay vòng đầu)

Duyệt vòng đầu, trừ dần; nếu ai đó thiếu ⇒ trả về ngay (không cần tổng). Nếu hết vòng mà vẫn còn phấn, lúc đó ta đã biết `S = k_ban_đầu − k_còn`, áp `mod` rồi duyệt lại.

In [10]:
def chalk_early(chalk, k):
    k0 = k
    for i, c in enumerate(chalk):
        if k < c:
            return i
        k -= c
    S = k0 - k                    # đã đi hết một vòng: S = tổng chalk
    r = k % S
    for i, c in enumerate(chalk):
        if r < c:
            return i
        r -= c
    return -1

for (chalk, k), want in examples:
    print(chalk_early(chalk, k), "kỳ vọng", want)

0 kỳ vọng 0
1 kỳ vọng 1


**Độ phức tạp:** Time `O(n)` worst (hai lượt), có thể `O(1)` nếu học sinh đầu đã thiếu. Space `O(1)`.

### 7.4 Cách 5 — Nhiều truy vấn

Tiền xử lý `pre` một lần; mỗi truy vấn `k` trả lời `bisect_right(pre, k % S)` trong `O(log n)`.

In [11]:
def chalk_many(chalk, ks):
    pre = list(accumulate(chalk)); S = pre[-1]
    return [bisect_right(pre, k % S) for k in ks]

print(chalk_many([3, 4, 1, 2], [0, 2, 3, 6, 7, 9, 10, 25, 10**9]))

[0, 0, 1, 1, 2, 3, 0, 1, 0]


## 8. TRACE từng bước

Trace Cách 1 (mô phỏng) cho ví dụ 2 và Cách 2 cho cùng input:

In [12]:
def trace_simulate(chalk, k):
    n = len(chalk); i = 0; step = 0
    while k >= chalk[i]:
        k -= chalk[i]; step += 1
        print(f"bước {step}: học sinh {i} dùng {chalk[i]} -> còn {k}")
        i = (i + 1) % n
    print(f"học sinh {i} cần {chalk[i]} > {k} -> thay phấn")
    return i

trace_simulate([3, 4, 1, 2], 25)

bước 1: học sinh 0 dùng 3 -> còn 22
bước 2: học sinh 1 dùng 4 -> còn 18
bước 3: học sinh 2 dùng 1 -> còn 17
bước 4: học sinh 3 dùng 2 -> còn 15
bước 5: học sinh 0 dùng 3 -> còn 12
bước 6: học sinh 1 dùng 4 -> còn 8
bước 7: học sinh 2 dùng 1 -> còn 7
bước 8: học sinh 3 dùng 2 -> còn 5
bước 9: học sinh 0 dùng 3 -> còn 2
học sinh 1 cần 4 > 2 -> thay phấn


1

In [13]:
def trace_mod(chalk, k):
    S = sum(chalk); r = k % S
    print(f"S = {S}, k = {k} = {k // S}·{S} + {r} -> bỏ qua {k // S} vòng, còn {r}")
    for i, c in enumerate(chalk):
        if r < c:
            print(f"học sinh {i}: cần {c} > {r} -> thay phấn"); return i
        r -= c
        print(f"học sinh {i}: dùng {c} -> còn {r}")

trace_mod([3, 4, 1, 2], 25)
trace_mod([5, 1, 5], 22)
trace_mod([3, 4, 1, 2], 10**9 + 7)

S = 10, k = 25 = 2·10 + 5 -> bỏ qua 2 vòng, còn 5
học sinh 0: dùng 3 -> còn 2
học sinh 1: cần 4 > 2 -> thay phấn
S = 11, k = 22 = 2·11 + 0 -> bỏ qua 2 vòng, còn 0
học sinh 0: cần 5 > 0 -> thay phấn
S = 10, k = 1000000007 = 100000000·10 + 7 -> bỏ qua 100000000 vòng, còn 7
học sinh 0: dùng 3 -> còn 4
học sinh 1: dùng 4 -> còn 0
học sinh 2: cần 1 > 0 -> thay phấn


2

Bảng tóm tắt Cách 3 với `pre = [3, 7, 8, 10]`:

| k | r = k mod 10 | bisect_right(pre, r) | học sinh |
|---|---|---|---|
| 25 | 5 | 1 | 1 |
| 9 | 9 | 3 | 3 |
| 7 | 7 | 2 | 2 (cần 1, còn 0) |
| 30 | 0 | 0 | 0 |

In [14]:
pre = [3, 7, 8, 10]
for k in [25, 9, 7, 30]:
    print(f"k={k}: r={k % 10}, học sinh {bisect_right(pre, k % 10)}, mô phỏng: {chalk_simulate([3, 4, 1, 2], k)}")

k=25: r=5, học sinh 1, mô phỏng: 1
k=9: r=9, học sinh 3, mô phỏng: 3
k=7: r=7, học sinh 2, mô phỏng: 2
k=30: r=0, học sinh 0, mô phỏng: 0


Trace Cách 4 (dừng sớm): khi phấn hết ngay trong vòng đầu, ta không cần tính tổng.

In [15]:
def trace_early(chalk, k):
    k0 = k
    for i, c in enumerate(chalk):
        if k < c:
            print(f"vòng đầu: học sinh {i} cần {c} > {k} -> trả về ngay (không cần tổng)"); return i
        k -= c
    S = k0 - k
    print(f"đi hết vòng đầu, biết S = {S}; còn {k} -> k mod S = {k % S}")
    r = k % S
    for i, c in enumerate(chalk):
        if r < c:
            print(f"học sinh {i} cần {c} > {r}"); return i
        r -= c

trace_early([3, 4, 1, 2], 5)
trace_early([3, 4, 1, 2], 25)

vòng đầu: học sinh 1 cần 4 > 2 -> trả về ngay (không cần tổng)
đi hết vòng đầu, biết S = 10; còn 15 -> k mod S = 5
học sinh 1 cần 4 > 2


1

## 9. Code nộp LeetCode

In [16]:
class Solution:
    def chalkReplacer(self, chalk: List[int], k: int) -> int:
        k %= sum(chalk)
        for i, c in enumerate(chalk):
            if k < c:
                return i
            k -= c
        return -1

sol = Solution()
for (chalk, k), want in examples:
    got = sol.chalkReplacer(chalk, k)
    print(got, "OK" if got == want else "SAI")

0 OK
1 OK


## 10. Vì sao thuật toán đúng

**Bổ đề (một vòng đầy đủ).** Nếu ở đầu một vòng còn `k ≥ S` viên, thì với mọi `i`, trước lượt của `i` còn `k − pre[i−1] ≥ S − pre[i−1] ≥ chalk[i]` ⇒ không ai thiếu, và vòng kết thúc với `k − S`. Ngược lại nếu `k < S`, sẽ có người thiếu trong vòng này (vì tổng nhu cầu của vòng là `S > k`).

**Kết luận.** Quy nạp: sau `q = ⌊k / S⌋` vòng đầy đủ, vòng tiếp theo bắt đầu với `r = k − qS = k mod S < S`. Học sinh thay phấn là `i` nhỏ nhất với `pre[i] > r` (lúc đó phấn còn `r − pre[i−1] < chalk[i]`). Cách 2 duyệt tìm đúng `i` này; Cách 3 tìm bằng `bisect_right` vì `pre` tăng nghiêm ngặt.

**Trường hợp `r = 0`:** học sinh 0 cần `chalk[0] ≥ 1 > 0` ⇒ học sinh 0 — đúng (ví dụ 1).

## 11. Phân tích Time / Space

Gọi `n ≤ 10⁵`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng | `O(k / avg)` tới `10⁹` | `O(1)` | kiểm chứng `k` nhỏ |
| 2. `mod` + một vòng | `O(n)` | `O(1)` | **nộp bài** |
| 3. Tiền tố + bisect | `O(n)` + `O(log n)` | `O(n)` | nhiều truy vấn |
| 4. Dừng sớm | `O(n)` worst | `O(1)` | khi thường hết phấn ở vòng đầu |
| 5. Nhiều truy vấn | `O(n + q log n)` | `O(n)` | |

In [17]:
random.seed(1)
chalk = [random.randint(1, 10**5) for _ in range(10**5)]
for k in [1, 10**9]:
    line = f"k={k:>10}:"
    res = []
    for name, fn in [("mod", chalk_mod), ("bisect", chalk_bisect), ("early", chalk_early), ("submit", Solution().chalkReplacer)]:
        t0 = time.perf_counter(); res.append(fn(chalk, k)); line += f" {name}={time.perf_counter() - t0:.4f}s"
    assert len(set(res)) == 1
    print(line, "-> học sinh", res[0])

k=         1: mod=0.0007s bisect=0.0039s early=0.0000s submit=0.0005s -> học sinh 0
k=1000000000: mod=0.0012s bisect=0.0031s early=0.0006s submit=0.0012s -> học sinh 20049


## 12. Bộ test

In [18]:
fns = [chalk_mod, chalk_bisect, chalk_early, lambda c, k: Solution().chalkReplacer(c, k)]

for (chalk, k), want in examples:
    assert chalk_simulate(chalk, k) == want
    for fn in fns:
        assert fn(chalk, k) == want

edge = [
    (([1], 1), 0), (([1], 10**9), 0), (([5], 4), 0),
    (([2, 3], 1), 0), (([2, 3], 2), 1), (([2, 3], 5), 0), (([2, 3], 4), 1),
    (([1, 1, 1], 2), 2), (([1, 1, 1], 3), 0),
    (([100000] * 3, 299999), 2),
]
for (chalk, k), want in edge:
    if k <= 10**6:                                     # mô phỏng chỉ khi k nhỏ (k = 1e9 sẽ rất chậm)
        assert chalk_simulate(chalk, k) == want, (chalk, k)
    for fn in fns:
        assert fn(chalk, k) == want, (fn, chalk, k)

random.seed(2024)
for _ in range(3000):
    n = random.randint(1, 8)
    chalk = [random.randint(1, random.choice([3, 20, 10**5])) for _ in range(n)]
    k = random.randint(1, min(10**9, 50 * sum(chalk)))
    want = chalk_simulate(chalk, k)
    for fn in fns:
        assert fn(chalk, k) == want, (fn, chalk, k)
print("ví dụ + edge + 3000 bộ ngẫu nhiên khớp mô phỏng")

ví dụ + edge + 3000 bộ ngẫu nhiên khớp mô phỏng


In [19]:
# nhiều truy vấn: khớp từng truy vấn riêng
random.seed(5)
for _ in range(100):
    chalk = [random.randint(1, 50) for _ in range(random.randint(1, 20))]
    ks = [random.randint(1, 10**9) for _ in range(30)]
    assert chalk_many(chalk, ks) == [Solution().chalkReplacer(chalk, k) for k in ks]
print("chalk_many khớp từng truy vấn")

chalk_many khớp từng truy vấn


In [20]:
# tính chất tuần hoàn: cộng thêm bội số của S vào k không đổi đáp án; và đáp án luôn là người "đầu tiên" thiếu
random.seed(6)
for _ in range(500):
    chalk = [random.randint(1, 30) for _ in range(random.randint(1, 12))]
    S = sum(chalk); k = random.randint(1, 10**6); q = random.randint(0, 1000)
    a = Solution().chalkReplacer(chalk, k)
    assert a == Solution().chalkReplacer(chalk, k + q * S)
    r = k % S
    assert sum(chalk[:a]) <= r < sum(chalk[:a + 1])       # mọi người trước a đủ phấn, a thiếu
print("tuần hoàn theo S và điều kiện 'người đầu tiên thiếu' đúng trên 500 bộ")

tuần hoàn theo S và điều kiện 'người đầu tiên thiếu' đúng trên 500 bộ


Hai tính chất trên là **đặc tả** của bài toán: (1) đáp án chỉ phụ thuộc `k mod S`; (2) người thay phấn `a` thỏa `pre[a−1] ≤ r < pre[a]`. Kiểm tra trực tiếp đặc tả là một cách test mạnh, không cần lời giải tham chiếu.

Edge `([2, 3], 5)`: `5 mod 5 = 0` ⇒ học sinh 0 (đã hết đúng một vòng, còn 0 viên). Edge `([2, 3], 2)`: học sinh 0 dùng 2, còn 0 < 3 ⇒ học sinh 1 — điều kiện "**nhỏ hơn nghiêm ngặt**": còn đúng bằng `chalk[i]` thì vẫn dùng được.

## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng `<=` thay vì `<` (còn đúng bằng nhu cầu vẫn bị coi là thiếu)

In [21]:
def wrong_le(chalk, k):
    k %= sum(chalk)
    for i, c in enumerate(chalk):
        if k <= c:                          # SAI: phải là k < c
            return i
        k -= c
    return -1

for chalk, k in [([2, 3], 2), ([3, 4, 1, 2], 7), ([5, 1, 5], 22)]:
    print(f"{chalk}, k={k}: sai = {wrong_le(chalk, k)} | đúng = {Solution().chalkReplacer(chalk, k)}")

[2, 3], k=2: sai = 0 | đúng = 1
[3, 4, 1, 2], k=7: sai = 1 | đúng = 2
[5, 1, 5], k=22: sai = 0 | đúng = 0


**Sửa:** `if k < c`.

### Lỗi 2 — Quên `mod` (chỉ duyệt một vòng)

In [22]:
def wrong_no_mod(chalk, k):
    for i, c in enumerate(chalk):
        if k < c:
            return i
        k -= c
    return -1                               # SAI: chưa xử lý các vòng tiếp theo

for (chalk, k), want in examples:
    print("sai =", wrong_no_mod(chalk, k), "| đúng =", want)

sai = -1 | đúng = 0
sai = -1 | đúng = 1


**Sửa:** `k %= sum(chalk)` trước khi duyệt.

### Lỗi 3 — Dùng `bisect_left` thay cho `bisect_right`

In [23]:
def wrong_bisect_left(chalk, k):
    from bisect import bisect_left
    pre = list(accumulate(chalk))
    return bisect_left(pre, k % pre[-1])    # SAI khi k % S trùng một tổng tiền tố

for chalk, k in [([3, 4, 1, 2], 7), ([3, 4, 1, 2], 13), ([2, 3], 2)]:
    print(f"{chalk}, k={k}: sai = {wrong_bisect_left(chalk, k)} | đúng = {Solution().chalkReplacer(chalk, k)}")

[3, 4, 1, 2], k=7: sai = 1 | đúng = 2
[3, 4, 1, 2], k=13: sai = 0 | đúng = 1
[2, 3], k=2: sai = 0 | đúng = 1


Khi `r = pre[i]`, học sinh `i` dùng hết đúng phần còn lại (không thiếu) ⇒ người thay là `i + 1`. `bisect_left` trả `i` — sai. **Sửa:** `bisect_right`.

### Lỗi 4 — Mô phỏng thẳng với `k = 10⁹` (TLE)

Minh họa bằng ước lượng thời gian (không chạy hết):

In [24]:
t0 = time.perf_counter(); chalk_simulate([1] * 10, 2 * 10**5); dt = time.perf_counter() - t0
print(f"2e5 bước mất {dt:.3f}s -> 1e9 bước ước tính ~{dt * 5000:.0f}s (quá giới hạn thời gian)")

2e5 bước mất 0.007s -> 1e9 bước ước tính ~36s (quá giới hạn thời gian)


**Sửa:** `mod` để bỏ qua các vòng đầy đủ.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Quá trình **lặp vòng** với lượng tiêu hao cố định mỗi vòng ⇒ `k mod (tổng một vòng)`.
- "Ai là người đầu tiên vượt ngưỡng" trên tổng dồn ⇒ tổng tiền tố + `bisect_right`.
- Chú ý "nhỏ hơn **nghiêm ngặt**" ⇒ `<` và `bisect_right`.
- Tổng có thể vượt `int32` ⇒ `long` ở C++/Java.
- Họ bài: 2073 *Time Needed to Buy Tickets*, 1823 *Find the Winner of the Circular Game*, 528 *Random Pick with Weight* (tiền tố + bisect).

## 15. Biến thể của bài

1. **Nhiều truy vấn `k`:** Cách 5.
2. **Thứ tự giao bài đảo chiều mỗi vòng** (0 → n−1, rồi n−1 → 0, …): một "chu kỳ" gồm hai vòng với tổng `2S`; xử lý `k mod 2S` rồi mô phỏng tối đa hai vòng.
3. **Mỗi học sinh tiêu phấn tăng dần theo vòng** (vòng `t` dùng `t · chalk[i]`): tổng sau `q` vòng là `S · q(q+1)/2` ⇒ nhị phân tìm số vòng đầy đủ.
4. **Trả về số phấn còn lại** khi học sinh đó phải thay.

In [25]:
def chalk_zigzag(chalk, k):
    """Biến thể 2: thứ tự 0..n-1, rồi n-1..0, lặp lại."""
    order = list(range(len(chalk))) + list(range(len(chalk) - 1, -1, -1))
    r = k % (2 * sum(chalk))
    for i in order:
        if r < chalk[i]:
            return i
        r -= chalk[i]

def chalk_zigzag_sim(chalk, k):
    order = list(range(len(chalk))) + list(range(len(chalk) - 1, -1, -1)); t = 0
    while True:
        i = order[t % len(order)]
        if k < chalk[i]: return i
        k -= chalk[i]; t += 1

def chalk_growing(chalk, k):
    """Biến thể 3: vòng t (t = 1, 2, ...) mỗi học sinh dùng t * chalk[i]."""
    S = sum(chalk)
    lo, hi = 0, 10**6
    while lo < hi:                               # q lớn nhất với S*q(q+1)/2 <= k
        mid = (lo + hi + 1) // 2
        if S * mid * (mid + 1) // 2 <= k: lo = mid
        else: hi = mid - 1
    r = k - S * lo * (lo + 1) // 2; t = lo + 1
    for i, c in enumerate(chalk):
        if r < t * c: return i
        r -= t * c

def chalk_growing_sim(chalk, k):
    t = 1
    while True:
        for i, c in enumerate(chalk):
            if k < t * c: return i
            k -= t * c
        t += 1

random.seed(3)
for _ in range(500):
    chalk = [random.randint(1, 9) for _ in range(random.randint(1, 6))]; k = random.randint(1, 500)
    assert chalk_zigzag(chalk, k) == chalk_zigzag_sim(chalk, k)
    assert chalk_growing(chalk, k) == chalk_growing_sim(chalk, k)
print("biến thể 2 (zigzag) và 3 (tiêu hao tăng dần) khớp mô phỏng")

biến thể 2 (zigzag) và 3 (tiêu hao tăng dần) khớp mô phỏng


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | đúng như đề | `O(k/avg)` | `O(1)` | kiểm chứng |
| 2. `mod` + một vòng | bỏ qua vòng đầy đủ | `O(n)` | `O(1)` | **nộp bài** |
| 3. Tiền tố + bisect | ngưỡng đầu tiên | `O(n)` + `O(log n)` | `O(n)` | truy vấn nhanh |
| 4. Dừng sớm | không cần tổng nếu hết sớm | `O(n)` | `O(1)` | |
| 5. Nhiều truy vấn | tiền tố dùng chung | `O(n + q log n)` | `O(n)` | nhiều `k` |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `chalkReplacer([1, 2, 3], 6)`, `chalkReplacer([1, 2, 3], 5)`, `chalkReplacer([10], 9)`?

**Bài 2 (tính tay):** `chalk = [4, 2, 7]`, `k = 100`: tính `S`, `k mod S`, và tìm học sinh.

**Bài 3 (điền TODO):** hoàn thiện `chalk_left(chalk, k)` trả về cặp `(học sinh, số phấn còn lại lúc đó)`.

**Bài 4 (tự viết lại):** viết lại Cách 3 **không dùng** `bisect` — tự cài tìm kiếm nhị phân "chỉ số đầu tiên có `pre[i] > r`".

**Bài 5 (mini challenge):** biến thể **học sinh `i` bị loại** sau khi thay phấn (không tham gia nữa) và giáo viên tiếp tục với phấn mới `k` (nạp lại đúng `k` ban đầu) cho tới khi còn 1 học sinh. Trả về học sinh cuối cùng. Viết mô phỏng và một lời giải nhanh hơn cho `n ≤ 2000`. (Biến thể tự đặt.)

In [26]:
def chalk_left(chalk, k):
    # TODO: r = k % S; duyệt tới học sinh đầu tiên có r < chalk[i]; trả (i, r)
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `S = 6`. (2) `6 mod 6 = 0` ⇒ học sinh 0; `5`: dùng 1 (còn 4), dùng 2 (còn 2), cần 3 > 2 ⇒ học sinh 2. (3) `[10], 9` ⇒ học sinh 0.

**Bài 2:** (1) `S = 13`. (2) `100 mod 13 = 9`. (3) Dùng 4 (còn 5), dùng 2 (còn 3), cần 7 > 3 ⇒ học sinh 2.

**Bài 3:** (1) Như Cách 2. (2) Trả `(i, r)` thay vì `i`. (3) Kiểm tra `r < chalk[i]`.

**Bài 4:** (1) `lo, hi = 0, n − 1`. (2) `if pre[mid] > r: hi = mid else lo = mid + 1`. (3) Trả `lo`.

**Bài 5:** (1) Mô phỏng: mỗi "lượt loại" chạy Cách 2 trên danh sách còn lại, bắt đầu từ người **sau** người bị loại. (2) Mỗi lượt `O(n)` ⇒ `O(n²)`. (3) Cẩn thận "bắt đầu từ đâu" sau khi loại — xoay danh sách.

## 19. Đáp án tham khảo

In [27]:
# Bài 1
assert [Solution().chalkReplacer([1, 2, 3], 6), Solution().chalkReplacer([1, 2, 3], 5), Solution().chalkReplacer([10], 9)] == [0, 2, 0]

# Bài 2
assert Solution().chalkReplacer([4, 2, 7], 100) == 2

# Bài 3
def chalk_left_ans(chalk, k):
    r = k % sum(chalk)
    for i, c in enumerate(chalk):
        if r < c:
            return i, r
        r -= c

# Bài 4
def chalk_manual_bs(chalk, k):
    pre = list(accumulate(chalk)); r = k % pre[-1]
    lo, hi = 0, len(pre) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if pre[mid] > r: hi = mid
        else: lo = mid + 1
    return lo

random.seed(221)
for _ in range(2000):
    chalk = [random.randint(1, 20) for _ in range(random.randint(1, 10))]; k = random.randint(1, 3000)
    i, r = chalk_left_ans(chalk, k)
    assert i == chalk_simulate(chalk, k) == chalk_manual_bs(chalk, k) and r < chalk[i]
print("Bài 1-4: OK; ví dụ 2 ->", chalk_left_ans([3, 4, 1, 2], 25))

Bài 1-4: OK; ví dụ 2 -> (1, 2)


In [28]:
# Bài 5 — loại dần (biến thể tự đặt)
def last_student_sim(chalk, k):
    alive = list(range(len(chalk))); start = 0
    while len(alive) > 1:
        rem = k; t = start
        while True:
            i = alive[t % len(alive)]
            if rem < chalk[i]:
                pos = t % len(alive); alive.pop(pos); start = pos      # người kế tiếp đứng ở vị trí pos
                break
            rem -= chalk[i]; t += 1
    return alive[0]

def last_student_fast(chalk, k):
    alive = list(range(len(chalk))); start = 0
    while len(alive) > 1:
        m = len(alive)
        order = alive[start:] + alive[:start]                      # bắt đầu từ start
        r = k % sum(chalk[i] for i in order)
        for off, i in enumerate(order):
            if r < chalk[i]:
                pos = (start + off) % m; break
            r -= chalk[i]
        alive.pop(pos); start = pos % len(alive)
    return alive[0]

random.seed(222)
for _ in range(400):
    chalk = [random.randint(1, 9) for _ in range(random.randint(1, 7))]; k = random.randint(1, 200)
    assert last_student_fast(chalk, k) == last_student_sim(chalk, k), (chalk, k)
print("Bài 5: khớp mô phỏng; ví dụ [3,4,1,2], k=25 ->", last_student_fast([3, 4, 1, 2], 25))

Bài 5: khớp mô phỏng; ví dụ [3,4,1,2], k=25 -> 2


## 20. Template nhớ nhanh + Checklist

```python
k %= sum(chalk)                  # bỏ qua mọi vòng đầy đủ
for i, c in enumerate(chalk):
    if k < c:                    # nhỏ hơn NGHIÊM NGẶT -> phải thay phấn
        return i
    k -= c
# Tiền tố: return bisect_right(list(accumulate(chalk)), k % S)
```

**Checklist trước khi submit:**
- [ ] `k %= S` trước khi duyệt (tránh mô phỏng `10⁹` bước).
- [ ] Điều kiện `k < chalk[i]` (nghiêm ngặt).
- [ ] Dùng `bisect_right` (không phải `bisect_left`) với tiền tố.
- [ ] `k mod S = 0` ⇒ học sinh 0.
- [ ] C++/Java: tổng dùng `long`.